In [73]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [74]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from SDRUtils.products.usd.usd_swaptions import USD_Swaptions
from SDRUtils.products._swaptions.pricer import (
    usd_swaption_straddle_pricer_from_row,
    usd_swaption_leg_pricer_from_row,
    usd_swaption_dealer_risk_reversal_skew_from_row,
    USDSwaptionStraddlePricerResult,
    USDSwaptionLegPricerResult,
	USDSwaptionDealerRiskReversalSkewResult,
    _compute_swaption_leg_greeks,
    USDSwaptionVerticalSpreadPricerResult,
    usd_swaption_vertical_spread_pricer_from_row
)

In [82]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

as_of = datetime.date(2026, 1, 21)
start = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 0, 0))
end = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 23, 59))

mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
# df

MERGING SLICES...: 100%|██████████| 2/2 [00:00<00:00, 42.37it/s]


In [85]:
# sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path)
sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=True, merge_package_legs=False)
# sdf

PRICING OUTRIGHTS...: 100%|██████████| 150/150 [00:01<00:00, 86.39it/s]


In [86]:
# .head(1).to_dict(orient="records")
temp = sdf[(sdf["trade_label"].str.contains("1Yx10Y"))][
    [
        "event_action",
        "trade_id",
        "execution_timestamp",
        "trade_label",
        "notional",
        "is_notional_capped",
        "strike",
        "premium",
        "platform_identifier",
        # "outright_atmf",
        # "outright_strike_offset_bps",
        # "outright_strike_offset_rounded_bps",
        # "outright_moneyness",
        "outright_bpvol_yr",
        # "outright_fwd_premium",
        # "outright_dv01",
        # "outright_vega01",
        # "outright_gamma01",
        # "outright_theta1d",
    ]
]
temp
# len(temp.drop_duplicates(subset=["trade_id"]))
# .head(3).to_dict(orient="records")

# sdf[sdf["ladder_notionals"].notna()].head(3)[
#
# ]

,event_action,trade_id,execution_timestamp,trade_label,notional,is_notional_capped,strike,premium,platform_identifier,outright_bpvol_yr
41,NEWT-TRAD,1802261889000001301,2026-01-21 13:38:56+00:00,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER...,75000000.0,False,0.042670,7.312500e+05,BILT,56.576347
59,NEWT-TRAD,1802852875000000401,2026-01-21 14:21:21+00:00,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y RECEI...,100000000.0,False,0.050180,7.750000e+04,BILT,NaN
60,NEWT-TRAD,1802852873000000201,2026-01-21 14:21:21+00:00,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER...,100000000.0,False,0.030180,7.750000e-02,BILT,NaN
64,NEWT-TRAD,1802922016000000501,2026-01-21 14:29:41+00:00,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER...,100000000.0,False,0.050150,4.050000e+05,TSEF,NaN
65,NEWT-TRAD,1802929775000000301,2026-01-21 14:30:42+00:00,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y RECEI...,100000000.0,False,0.030150,3.200000e+05,TSEF,NaN
66,NEWT-TRAD,1802934489000000501,2026-01-21 14:31:02+00:00,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER...,18000000.0,False,0.040150,4.499994e+05,TSEF,NaN
67,NEWT-TRAD,1802938757000000501,2026-01-21 14:31:33+00:00,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y RECEI...,18000000.0,False,0.040150,4.499994e+05,TSEF,NaN
68,NEWT-TRAD,1802943221000000301,2026-01-21 14:31:50+00:00,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER...,250000000.0,True,0.050150,1.000000e+06,TSEF,NaN
69,NEWT-TRAD,1802943223000000501,2026-01-21 14:31:59+00:00,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y RECEI...,250000000.0,True,0.030150,8.000000e+05,TSEF,NaN
70,NEWT-TRAD,1802962537000000101,2026-01-21 14:33:22+00:00,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER...,88000000.0,False,0.040150,2.199991e+06,TSEF,NaN


In [89]:
# cols = [
#     "Dissemination Identifier",
#     "Original Dissemination Identifier",
#     "Action type",
#     "Event type",
#     "Platform identifier",
#     "Execution Timestamp",
#     "Notional amount-Leg 1",
#     "Option Premium Amount",
#     "Strike Price",
# ]

df[(df["Dissemination Identifier"] == "1802943221000000301") | (df["Original Dissemination Identifier"] == "1802943221000000301")].to_dict(orient="records")

# df[(df["Action type"] == "NEWT") & (df["Strike Price"] >= 0.050150) & (df["Strike Price"] <= 0.050180)][cols]
# .tail(3)

[{'Dissemination Identifier': '1802943221000000301',
  'Original Dissemination Identifier': '',
  'Action type': 'NEWT',
  'Event type': 'TRAD',
  'Event timestamp': Timestamp('2026-01-21 14:31:50+0000', tz='UTC'),
  'Amendment indicator': None,
  'Asset Class': 'IR',
  'Product name': None,
  'Cleared': 'N',
  'Mandatory clearing indicator': False,
  'Execution Timestamp': Timestamp('2026-01-21 14:31:50+0000', tz='UTC'),
  'Effective Date': Timestamp('2026-01-21 00:00:00'),
  'Expiration Date': Timestamp('2027-01-21 00:00:00'),
  'Maturity date of the underlier': datetime.date(2037, 1, 25),
  'Non-standardized term indicator': False,
  'Platform identifier': 'TSEF',
  'Prime brokerage transaction indicator': False,
  'Block trade election indicator': False,
  'Large notional off-facility swap election indicator': None,
  'Notional amount-Leg 1': '250,000,000+',
  'Notional amount-Leg 2': '250,000,000+',
  'Notional currency-Leg 1': 'USD',
  'Notional currency-Leg 2': 'USD',
  'Notiona

In [44]:
df[df["Dissemination Identifier"] == "1802994050000000301"][
    cols
]

,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Platform identifier,Execution Timestamp,Notional amount-Leg 1,Option Premium Amount,Strike Price
12139,1802994050000000301,,NEWT,TRAD,TSEF,2026-01-21 14:37:37+00:00,"250,000,000+","1,012,500",0.05015


In [73]:
import ujson as json


def format_swaption_pricing_results(
    results: USDSwaptionStraddlePricerResult | USDSwaptionLegPricerResult | USDSwaptionDealerRiskReversalSkewResult,
):
    if isinstance(results, USDSwaptionDealerRiskReversalSkewResult):
        output = {
            "trade": results.trade_label,
            "atm_strike": results.atm_strike * 100,
            "otm_payer_strike": results.otm_payer_strike * 100,
            "otm_receiver_strike": results.otm_receiver_strike * 100,
            "wing_strike_width": results.wing_strike_width,
            "atm_bpvol": results.atm_bpvol_yr,
            "otm_payer_bpvol": results.otm_payer_bpvol_yr,
            "otm_receiver_bpvol": results.otm_receiver_bpvol_yr,
            "payer_skew_bpvol_yr": results.payer_skew_bpvol_yr,
            "receiver_skew_bpvol_yr": results.receiver_skew_bpvol_yr,
            "skew_bpvol": results.skew_bpvol_yr,
            "atm_notional": results.atm_notional,
            "wing_notional": results.wing_notional,
            "otm_payer_vega01": results.otm_payer_vega01,
            "otm_receiver_vega01": results.otm_receiver_vega01,
            "dv01": results.dv01,
            "gamma01": results.gamma01,
            "vega01": results.vega01,
            "theta1d": results.theta1d,
            "wing_dv01": results.wing_dv01
        }
    elif isinstance(results, USDSwaptionVerticalSpreadPricerResult):
        output = {
            "trade": results.trade_label,
            "spread_type": results.spread_type,
            # Strikes
            "atm_strike": results.atm_strike * 100,
            "otm_strike": results.otm_strike * 100,
            "strike_width_bps": results.strike_width_bps,
            "atm_strike_offset": results.atm_strike_offset,
            "otm_strike_offset": results.otm_strike_offset,
            # Vols
            "atm_bpvol": results.atm_bpvol_yr,
            "otm_bpvol": results.otm_bpvol_yr,
            "vol_spread_bpvol": results.vol_spread_bpvol_yr,
            # Notionals
            "atm_notional": results.atm_notional,
            "otm_notional": results.otm_notional,
            "notional_ratio": results.notional_ratio,
            # Premiums
            "net_premium": results.net_premium,
            "atm_premium": results.atm_premium,
            "otm_premium": results.otm_premium,
            # ATM leg Greeks
            "atm_dv01": results.atm_dv01,
            "atm_gamma01": results.atm_gamma01,
            "atm_vega01": results.atm_vega01,
            
            "atm_theta1d": results.atm_theta1d,
            # OTM leg Greeks
            "otm_dv01": results.otm_dv01,
            "otm_gamma01": results.otm_gamma01,
            "otm_vega01": results.otm_vega01,
            "otm_theta1d": results.otm_theta1d,
            # Aggregate Greeks
            "dv01": results.dv01,
            "gamma01": results.gamma01,
            "vega01": results.vega01,
            "theta1d": results.theta1d,
        }
    else:
        output = {
            "trade": results.trade_label,
            "prem": (results.fwd_prem / results.notional) * 10_000,
            "bpvol": results.bpvol_yr,
            "bpvol_day": results.bpvol_yr / np.sqrt(252),
            "dv01": results.dv01,
            "gamma01": results.gamma01,
            "vega01": results.vega01,
            "theta1d": results.theta1d,
        }

    print(json.dumps(output, indent=4))

In [75]:
# format_swaption_pricing_results(usd_swaption_leg_pricer_from_row(sdf.loc[168], pricer))
format_swaption_pricing_results(usd_swaption_straddle_pricer_from_row(sdf.loc[490], pricer))

{
    "trade": "USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER EURO VANILLA PHYS",
    "prem": 480.0,
    "bpvol": 72.25211145222885,
    "bpvol_day": 4.5514552048076045,
    "dv01": 852.9995830946937,
    "gamma01": 663.3399956226453,
    "vega01": 64548.32454711436,
    "theta1d": -6340.930749212392
}


In [18]:
# format_swaption_pricing_results(usd_swaption_straddle_pricer_from_row(sdf.loc[199], pricer))
# format_swaption_pricing_results(usd_swaption_leg_pricer_from_row(sdf.loc[96], pricer))

# format_swaption_pricing_results(usd_swaption_dealer_risk_reversal_skew_from_row(risk_reversal_row=sdf.loc[286], pricer=pricer))

In [53]:

row = sdf.loc[168]

display(row.to_dict())
_compute_swaption_leg_greeks(
	pricer,
	row["expiration_date"],
	row["underlying_expiration_date"],
	row["strike"],
	row["notional"],
	row["premium"],
	"receiver" if "rec" in row["product_type"].lower() else "payer",
)

{'event_action': 'NEWT-TRAD',
 'trade_id': '1745734985000000401',
 'execution_timestamp': Timestamp('2026-01-15 20:56:16+0000', tz='UTC'),
 'effective_date': Timestamp('2026-01-15 00:00:00'),
 'expiration_date': Timestamp('2028-01-10 00:00:00'),
 'product_type': 'SWAPTION_PAYER',
 'trade_label': 'USD-SOFR-OIS Compound 1Y CONSTANT 2Yx30Y PAYER EURO VANILLA PHYS',
 'notional': 100000000.0,
 'notional_currency': 'USD',
 'is_notional_capped': False,
 'package_type': 'SWAPTION',
 'package_id': None,
 'package_legs': None,
 'underlying_expiration_date': Timestamp('2058-01-12 00:00:00'),
 'tenor_years': 30.027397260273972,
 'tenor_label': '30Y',
 'forward_start_years': 1.9863013698630136,
 'forward_label': '2Y',
 'premium': 450000.0,
 'exercise_style': 'EUROPEAN',
 'strike': 0.05234,
 'upi_underlier_name': 'NA/Swap OIS USD',
 'unique_product_identifier': 'QZZLNQ2D4JQT',
 'platform_identifier': 'BILT',
 'cleared': 'N',
 'package_indicator': False,
 'package_transaction_price': '',
 'option_pre

_SwaptionLegGreeks(bpvol_yr=50.80008511489668, dv01=13146.336552531715, gamma01=394.24023436883004, vega01=34351.294088425864, theta1d=1203.1005319558317, strike_offset=100)